In [ ]:
BATCH_SIZE = 32


In [ ]:
import sys
sys.path.append('../../../')
from models.value_cnn import ValueCNNDecentralized, ValueCNNCentralized
from models.cnn import CNNDecentralized
import torch
from tadd_helpers.env_functions import State
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [ ]:
def get_decentralized_reward_when_picked(picked: bool, picker_index, picker_pos, all_agents) -> dict[int, float]:
    """Return decentralized reward of all agents

    Args:
        picked (bool): Whether an agent has picked an apple
        picker_index: _description_
        picker_pos: _description_
        all_agents: dict[int, tuple]: Mapping from agent index to (x, y) position
    """
    res: dict[int, float] = {}
    
    if not picked:
        for agent_idx in all_agents.keys():
            res[agent_idx] = 0.0
        return res

    res[picker_index] = -1

    # Calculate distances from ALL agents to the picker
    all_agent_positions = np.array(list(all_agents.values()))
    distances = np.linalg.norm(all_agent_positions - np.array(picker_pos), axis=1)
    sum_of_distances = np.sum(distances)
    
    for agent_idx in all_agents.keys():
        if agent_idx == picker_index:
            continue
        agent_pos = all_agents[agent_idx]
        agent_distance = np.linalg.norm(np.array(agent_pos) - np.array(picker_pos))
        if sum_of_distances == 0:
            res[agent_idx] = 0.0
        else:
            res[agent_idx] = 2 * agent_distance / sum_of_distances
    return res

def get_picker_index_and_pos_from_state(state: State):
    """Extract the picker index and position from the state. 

    Args:
        state (State): The current environment state
    Returns:
        picked: bool: Whether an agent has picked an apple
        picker_index (int): The index of the picker agent
        picker_pos (tuple): The (x, y) position of the picker agent 
    """
    picked = False
    picker_index = -1
    picker_pos = (-1, -1)
    for agent_idx, agent_pos in state._agents.items():
        # check if the agent_pos is in the apples nd array
        if state._apples[agent_pos] >= 1:
            picked = True
            picker_index = agent_idx
            picker_pos = agent_pos
            break
    return picked, picker_index, picker_pos

def get_decentralized_reward_from_state(state: State) -> dict[int, float]:
    """Calculate the decentralized reward for the self-agent based on the state.

    Args:
        state (State): The current environment state
    Returns:
        dict[int, float]: The decentralized rewards for all agents
    """
    picked, picker_index, picker_pos = get_picker_index_and_pos_from_state(state)
    all_agents = state._agents
    rewards = get_decentralized_reward_when_picked(picked, picker_index, picker_pos, all_agents)
    return rewards

In [ ]:
decen_path = "reward_model_6x6_2a"
decen_model_paths = []
for i in range(2):
    decen_model_paths.append(f"{decen_path}/model_{i}.pt")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
decen_cnns: list[CNNDecentralized] = []
for i in range(len(decen_model_paths)):
    model_path = decen_model_paths[i]
    cnn = CNNDecentralized(6, 6, 0.0001, 128, 2, [16, 32], 3)
    
    state_dict = torch.load(model_path, map_location=device)
    cnn.load_state_dict(state_dict)
    # --------------------
    
    # 3. Explicitly move the entire model to the chosen device
    cnn.to(device)

    cnn.eval()
    decen_cnns.append(cnn)


In [ ]:
states: list[State] = []
states_file_path = "centralized_model6x6_agents2/trained_states_centralized.pkl"
with open(states_file_path, "rb") as f:
    states = pickle.load(f)
print(states[:3])

In [ ]:
print(len(states))
states_to_eval = states

In [ ]:
# In pred_6x6_full_trajectory_ir.ipynb

# --- This is the corrected processing loop ---
all_reward_results = []

# Using total=len(...) will give you a proper progress bar
for t, state in tqdm(enumerate(states_to_eval), total=len(states_to_eval), desc="Predicting Immediate Rewards"):
    
    # --- 1. Get the Ground Truth Immediate Rewards ---
    true_rewards = get_decentralized_reward_from_state(state)
    true_reward_agent0 = true_rewards.get(0, 0.0)
    true_reward_agent1 = true_rewards.get(1, 0.0)

    # --- 2. Get the Predicted Immediate Rewards from the trained models ---
    predicted_rewards = {}
    for i in range(2): # Assuming NUM_AGENTS = 2
        cnn = decen_cnns[i]
        
        # The raw_dict can be created once
        raw_dict = {"agents": state.agents, "apples": state.apples}
        
        # Get the reward prediction for agent i
        ir_pred = cnn.get_model_reward_prediction_from_raw(
            raw_dict, 
            agent_pos=state.agent_position(i)
        ).item()
        predicted_rewards[i] = ir_pred
        
    pred_reward_agent0 = predicted_rewards.get(0, 0.0)
    pred_reward_agent1 = predicted_rewards.get(1, 0.0)

    # --- 3. Store all relevant data for this state ---
    result_info = {
        "state_index": t, # Use the original index as a unique key
        "timestep": t / 2,
        "true_reward_A0": true_reward_agent0,
        "pred_reward_A0": pred_reward_agent0,
        "error_A0": abs(true_reward_agent0 - pred_reward_agent0),
        "true_reward_A1": true_reward_agent1,
        "pred_reward_A1": pred_reward_agent1,
        "error_A1": abs(true_reward_agent1 - pred_reward_agent1),
        "total_reward_error": abs((true_reward_agent0 + true_reward_agent1) - (pred_reward_agent0 + pred_reward_agent1)),
        "state_object": state
    }
    all_reward_results.append(result_info)


In [ ]:

# --- Create the DataFrame ---
df_ir = pd.DataFrame(all_reward_results)

# --- Save the DataFrame to a new pickle file ---
with open("predicted_trajectory_ir_6x6_agents2.pkl", "wb") as f:
    pickle.dump(df_ir, f)

# --- Display the head to verify ---
print("\nImmediate Reward (IR) Analysis DataFrame:")
print(df_ir.head())